# 10 — Grid Search Mean Temp / 10 Grid Search Mean Temp

**Chapter 11 — File 10 of 12 / 第11章 — 第10个文件（共12个）**

---

## Summary / 总结

This script demonstrates **grid search simple forecast for monthly mean temperature**.

本脚本演示 **grid search simple forecast for monthly mean temperature**。

---
## Step 1 — grid search simple forecast for monthly mean temperature

In [ ]:
from math import sqrt
from numpy import mean
from numpy import median
from multiprocessing import cpu_count
from joblib import Parallel
from joblib import delayed
from warnings import catch_warnings
from warnings import filterwarnings
from sklearn.metrics import mean_squared_error
from pandas import read_csv

---
## Step 2 — one-step simple forecast

In [ ]:
def simple_forecast(history, config):
	n, offset, avg_type = config

---
## Step 3 — persist value, ignore other config

In [ ]:
if avg_type == 'persist':
		return history[-n]

---
## Step 4 — collect values to average

In [ ]:
values = list()
	if offset == 1:
		values = history[-n:]
	else:

---
## Step 5 — skip bad configs

In [ ]:
if n*offset > len(history):
			raise Exception('Config beyond end of data: %d %d' % (n,offset))

---
## Step 6 — try and collect n values using offset

In [ ]:
for i in range(1, n+1):
			ix = i * offset
			values.append(history[-ix])

---
## Step 7 — check if we can average

In [ ]:
if len(values) < 2:
		raise Exception('Cannot calculate average')

---
## Step 8 — mean of last n values

In [ ]:
if avg_type == 'mean':
		return mean(values)

---
## Step 9 — median of last n values

In [ ]:
return median(values)

---
## Step 10 — root mean squared error or rmse

In [ ]:
def measure_rmse(actual, predicted):
	return sqrt(mean_squared_error(actual, predicted))

---
## Step 11 — split a univariate dataset into train/test sets

In [ ]:
def train_test_split(data, n_test):
	return data[:-n_test], data[-n_test:]

---
## Step 12 — walk-forward validation for univariate data

In [ ]:
def walk_forward_validation(data, n_test, cfg):
	predictions = list()

---
## Step 13 — split dataset

In [ ]:
train, test = train_test_split(data, n_test)

---
## Step 14 — seed history with training dataset

In [ ]:
history = [x for x in train]

---
## Step 15 — step over each time-step in the test set

In [ ]:
for i in range(len(test)):

---
## Step 16 — fit model and make forecast for history

In [ ]:
yhat = simple_forecast(history, cfg)

---
## Step 17 — store forecast in list of predictions

In [ ]:
predictions.append(yhat)

---
## Step 18 — add actual observation to history for the next loop

In [ ]:
history.append(test[i])

---
## Step 19 — estimate prediction error

In [ ]:
error = measure_rmse(test, predictions)
	return error

---
## Step 20 — score a model, return None on failure

In [ ]:
def score_model(data, n_test, cfg, debug=False):
	result = None

---
## Step 21 — convert config to a key

In [ ]:
key = str(cfg)

---
## Step 22 — show all warnings and fail on exception if debugging

In [ ]:
if debug:
		result = walk_forward_validation(data, n_test, cfg)
	else:

---
## Step 23 — one failure during model validation suggests an unstable config

In [ ]:
try:

---
## Step 24 — never show warnings when grid searching, too noisy

In [ ]:
with catch_warnings():
				filterwarnings("ignore")
				result = walk_forward_validation(data, n_test, cfg)
		except:
			error = None

---
## Step 25 — check for an interesting result

In [ ]:
if result is not None:
		print(' > Model[%s] %.3f' % (key, result))
	return (key, result)

---
## Step 26 — grid search configs

In [ ]:
def grid_search(data, cfg_list, n_test, parallel=True):
	scores = None
	if parallel:

---
## Step 27 — execute configs in parallel

In [ ]:
executor = Parallel(n_jobs=cpu_count(), backend='multiprocessing')
		tasks = (delayed(score_model)(data, n_test, cfg) for cfg in cfg_list)
		scores = executor(tasks)
	else:
		scores = [score_model(data, n_test, cfg) for cfg in cfg_list]

---
## Step 28 — remove empty results

In [ ]:
scores = [r for r in scores if r[1] != None]

---
## Step 29 — sort configs by error, asc

In [ ]:
scores.sort(key=lambda tup: tup[1])
	return scores

---
## Step 30 — create a set of simple configs to try

In [ ]:
def simple_configs(max_length, offsets=[1]):
	configs = list()
	for i in range(1, max_length+1):
		for o in offsets:
			for t in ['persist', 'mean', 'median']:
				cfg = [i, o, t]
				configs.append(cfg)
	return configs

if __name__ == '__main__':

---
## Step 31 — define dataset

In [ ]:
series = read_csv('monthly-mean-temp.csv', header=0, index_col=0)
	data = series.values

---
## Step 32 — data split

In [ ]:
n_test = 12

---
## Step 33 — model configs

In [ ]:
max_length = len(data) - n_test
	cfg_list = simple_configs(max_length, offsets=[1,12])

---
## Step 34 — grid search

In [ ]:
scores = grid_search(data, cfg_list, n_test)
	print('done')

---
## Step 35 — list top 3 configs

In [ ]:
for cfg, error in scores[:3]:
		print(cfg, error)

---
## Learning Notes / 学习笔记

- **概念**: grid search simple forecast for monthly mean temperature 是机器学习中的常用技术。  
  *grid search simple forecast for monthly mean temperature is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Grid Search Mean Temp / 10 Grid Search Mean Temp
# Complete Code / 完整代码
# ===============================

# grid search simple forecast for monthly mean temperature
from math import sqrt
from numpy import mean
from numpy import median
from multiprocessing import cpu_count
from joblib import Parallel
from joblib import delayed
from warnings import catch_warnings
from warnings import filterwarnings
from sklearn.metrics import mean_squared_error
from pandas import read_csv

# one-step simple forecast
def simple_forecast(history, config):
	n, offset, avg_type = config
	# persist value, ignore other config
	if avg_type == 'persist':
		return history[-n]
	# collect values to average
	values = list()
	if offset == 1:
		values = history[-n:]
	else:
		# skip bad configs
		if n*offset > len(history):
			raise Exception('Config beyond end of data: %d %d' % (n,offset))
		# try and collect n values using offset
		for i in range(1, n+1):
			ix = i * offset
			values.append(history[-ix])
	# check if we can average
	if len(values) < 2:
		raise Exception('Cannot calculate average')
	# mean of last n values
	if avg_type == 'mean':
		return mean(values)
	# median of last n values
	return median(values)

# root mean squared error or rmse
def measure_rmse(actual, predicted):
	return sqrt(mean_squared_error(actual, predicted))

# split a univariate dataset into train/test sets
def train_test_split(data, n_test):
	return data[:-n_test], data[-n_test:]

# walk-forward validation for univariate data
def walk_forward_validation(data, n_test, cfg):
	predictions = list()
	# split dataset
	train, test = train_test_split(data, n_test)
	# seed history with training dataset
	history = [x for x in train]
	# step over each time-step in the test set
	for i in range(len(test)):
		# fit model and make forecast for history
		yhat = simple_forecast(history, cfg)
		# store forecast in list of predictions
		predictions.append(yhat)
		# add actual observation to history for the next loop
		history.append(test[i])
	# estimate prediction error
	error = measure_rmse(test, predictions)
	return error

# score a model, return None on failure
def score_model(data, n_test, cfg, debug=False):
	result = None
	# convert config to a key
	key = str(cfg)
	# show all warnings and fail on exception if debugging
	if debug:
		result = walk_forward_validation(data, n_test, cfg)
	else:
		# one failure during model validation suggests an unstable config
		try:
			# never show warnings when grid searching, too noisy
			with catch_warnings():
				filterwarnings("ignore")
				result = walk_forward_validation(data, n_test, cfg)
		except:
			error = None
	# check for an interesting result
	if result is not None:
		print(' > Model[%s] %.3f' % (key, result))
	return (key, result)

# grid search configs
def grid_search(data, cfg_list, n_test, parallel=True):
	scores = None
	if parallel:
		# execute configs in parallel
		executor = Parallel(n_jobs=cpu_count(), backend='multiprocessing')
		tasks = (delayed(score_model)(data, n_test, cfg) for cfg in cfg_list)
		scores = executor(tasks)
	else:
		scores = [score_model(data, n_test, cfg) for cfg in cfg_list]
	# remove empty results
	scores = [r for r in scores if r[1] != None]
	# sort configs by error, asc
	scores.sort(key=lambda tup: tup[1])
	return scores

# create a set of simple configs to try
def simple_configs(max_length, offsets=[1]):
	configs = list()
	for i in range(1, max_length+1):
		for o in offsets:
			for t in ['persist', 'mean', 'median']:
				cfg = [i, o, t]
				configs.append(cfg)
	return configs

if __name__ == '__main__':
	# define dataset
	series = read_csv('monthly-mean-temp.csv', header=0, index_col=0)
	data = series.values
	# data split
	n_test = 12
	# model configs
	max_length = len(data) - n_test
	cfg_list = simple_configs(max_length, offsets=[1,12])
	# grid search
	scores = grid_search(data, cfg_list, n_test)
	print('done')
	# list top 3 configs
	for cfg, error in scores[:3]:
		print(cfg, error)

---

➡️ **Next / 下一步**: File 11 of 12